# Regresión Multinomial Bayesiana — Compradores del Movistar Arena (5 variables + clase de referencia)

Este cuaderno implementa una **regresión logística multinomial bayesiana** con PyMC para clasificar compradores de boletas del concierto de Miguel Amezquita en el Movistar Arena dentro de tres perfiles:

| Clase | Código | Descripción |
|-------|--------|-------------|
| `Planner` | 0 | Compra con semanas de anticipación, comportamiento planificado |
| `In-Between` | 1 | Ni planificador ni compulsivo, grupo indeciso |
| `Last-Minute` | 2 | Compra a último momento, impulsado por urgencia |

**Predictores en esta versión:**
- `Age`
- `Num_Tickets_Purchased`
- `Concession_Purchases`
- `fan_int` (derivada de `Fan_Mailing_List`)
- `seat_lower` (derivada de `Seat_Location`)

**Corrección aplicada (identificabilidad):**
- En lugar de estimar logits libres para las 3 clases, fijamos `Planner` como clase de referencia.
- Se estima solo:
  - `alpha_raw` de tamaño 2 (`In-Between`, `Last-Minute`)
  - `beta_raw` de tamaño `n_pred × 2`
- Luego concatenamos ceros para la clase base (`Planner`) y reconstruimos:
  - `alpha = [0, alpha_raw]`
  - `beta = [0_col, beta_raw]`
- Así evitamos la dirección plana del softmax y anclamos la inferencia.

**Modelo matemático (multinomial con referencia):**
$$P(Y_i = c \mid X_i) = \frac{\exp(\eta_{i,c})}{\sum_{j=0}^{2}\exp(\eta_{i,j})}$$
con
$$\eta_{i,0}=0 \quad (\text{Planner, referencia}), \qquad \eta_{i,c}=\alpha_c + X_i^\prime\beta_c,\ c\in\{1,2\}$$

**Facultad de Ciencia de Datos — Universidad Externado de Colombia**


## 0. Instalación de dependencias (Google Colab)

Si estás en Google Colab, descomenta la siguiente celda para instalar las librerías necesarias.

In [ ]:
# Descomenta si estás en Google Colab
# !pip install pymc arviz openpyxl scikit-learn -q

## 1. Importaciones y configuración

Importamos todas las librerías necesarias al inicio del cuaderno, siguiendo el estilo del profesor.

In [ ]:
# numpy: operaciones numéricas vectorizadas (arrays, matrices)
import numpy as np

# pandas: lectura y manipulación de datos tabulares (DataFrames)
import pandas as pd

# matplotlib: librería base de visualización en Python
import matplotlib.pyplot as plt

# seaborn: visualización estadística de alto nivel, construida sobre matplotlib
import seaborn as sns

# pymc: librería principal para modelado probabilístico bayesiano con MCMC
import pymc as pm

# arviz: librería para análisis e interpretación de trazas bayesianas (diagnósticos y plots)
import arviz as az

# pytensor tensor ops: concatenaciones/ceros para parametrización con clase de referencia
import pytensor.tensor as pt

# StandardScaler: estandariza variables (media=0, desv=1) para mejorar la convergencia MCMC
from sklearn.preprocessing import StandardScaler, LabelEncoder

# confusion_matrix, classification_report: métricas de evaluación del clasificador
from sklearn.metrics import confusion_matrix, classification_report

# train_test_split: dividir datos en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# warnings: suprimir advertencias menores que no afectan los resultados
import warnings
warnings.filterwarnings('ignore')

# os: manejo de directorios del sistema (para crear carpetas de figuras)
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

IMG_DIR = '../img/resultados_movistar_5vars_ref'
os.makedirs(IMG_DIR, exist_ok=True)

print(f'PyMC versión:  {pm.__version__}')
print(f'ArviZ versión: {az.__version__}')
print(f'NumPy versión: {np.__version__}')


## 2. Carga y EDA

### 2.1 Lectura del dataset

El archivo `movistar.xlsx` contiene 1000 registros de compradores del concierto. Cada fila es un cliente único con sus características de compra.

In [ ]:
# Ruta al archivo de datos — ajusta si ejecutas desde Google Colab
DATA_PATH = '../data/movistar.xlsx'

# pd.read_excel: lee el archivo xlsx y lo convierte en un DataFrame de pandas
# Cada columna del Excel se convierte en una Serie con su tipo de dato automáticamente
df = pd.read_excel(DATA_PATH)

# Información básica del dataset
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nTipos de variables:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

# Mostrar los primeros 5 registros
df.head()

### 2.2 Estadísticas descriptivas del modelo de 5 variables


In [ ]:
# Estadísticas generales de las variables seleccionadas
print('=== Estadísticas globales ===')
display(df[['Age', 'Num_Tickets_Purchased', 'Concession_Purchases']].describe().round(2))

print('\n=== Media por tipo de cliente ===')
display(
    df.groupby('Customer_Type')[[
        'Age', 'Num_Tickets_Purchased', 'Concession_Purchases'
    ]].mean().round(2)
)

print('\n=== Proporción de Fan_Mailing_List por tipo de cliente ===')
display(
    pd.crosstab(df['Customer_Type'], df['Fan_Mailing_List'], normalize='index').round(3)
)

print('\n=== Proporción de Seat_Location por tipo de cliente ===')
display(
    pd.crosstab(df['Customer_Type'], df['Seat_Location'], normalize='index').round(3)
)

print('\n=== Distribución de Customer_Type ===')
print(df['Customer_Type'].value_counts())
print(f'\nPorcentajes:')
print((df['Customer_Type'].value_counts(normalize=True) * 100).round(1))


### 2.3 Distribución de la variable objetivo

In [ ]:
# Conteo de clientes por tipo: verificamos si el dataset está balanceado
# Un dataset desbalanceado puede sesgar el modelo hacia la clase mayoritaria
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Countplot: barras con el conteo de cada categoría
conteo = df['Customer_Type'].value_counts()
axes[0].bar(conteo.index, conteo.values, color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='white')
axes[0].set_title('Conteo por tipo de cliente')
axes[0].set_xlabel('Customer_Type')
axes[0].set_ylabel('Número de clientes')
for i, (tipo, val) in enumerate(conteo.items()):
    axes[0].text(i, val + 5, str(val), ha='center', fontweight='bold')

# Pie chart: proporción de cada clase
axes[1].pie(
    conteo.values, labels=conteo.index,
    autopct='%1.1f%%', colors=['#2196F3', '#FF9800', '#4CAF50'],
    startangle=90
)
axes[1].set_title('Proporción por tipo de cliente')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/01_distribucion_clases.png', bbox_inches='tight')
plt.show()
print('Guardado: 01_distribucion_clases.png')

### 2.4 Boxplots de predictores por tipo de cliente


In [ ]:
# Variables numéricas continuas a comparar por clase
vars_continuas = [
    'Age',
    'Num_Tickets_Purchased',
    'Concession_Purchases'
]

orden_clases = ['Planner', 'In-Between', 'Last-Minute']
paleta = {'Planner': '#2196F3', 'In-Between': '#FF9800', 'Last-Minute': '#4CAF50'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, var in enumerate(vars_continuas):
    sns.boxplot(
        x='Customer_Type', y=var, data=df,
        order=orden_clases, palette=paleta, ax=axes[i]
    )
    axes[i].set_title(f'{var} por tipo de cliente')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=15)

# Fan_Mailing_List por clase
fan_plot = (
    pd.crosstab(df['Customer_Type'], df['Fan_Mailing_List'], normalize='index')
    .reindex(orden_clases)
)
fan_plot.plot(
    kind='bar', stacked=True, ax=axes[3],
    color=['#BDBDBD', '#9C27B0'], edgecolor='white'
)
axes[3].set_title('Fan_Mailing_List por tipo de cliente')
axes[3].set_xlabel('')
axes[3].set_ylabel('Proporción')
axes[3].tick_params(axis='x', rotation=15)
axes[3].legend(title='Fan')

# Seat_Location por clase
seat_plot = (
    pd.crosstab(df['Customer_Type'], df['Seat_Location'], normalize='index')
    .reindex(orden_clases)
)
seat_plot.plot(
    kind='bar', stacked=True, ax=axes[4],
    color=['#26A69A', '#90A4AE'], edgecolor='white'
)
axes[4].set_title('Seat_Location por tipo de cliente')
axes[4].set_xlabel('')
axes[4].set_ylabel('Proporción')
axes[4].tick_params(axis='x', rotation=15)
axes[4].legend(title='Ubicación')

axes[5].set_visible(False)

plt.suptitle('Distribución de variables por tipo de comprador', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/02_boxplots_por_clase.png', bbox_inches='tight')
plt.show()
print('Guardado: 02_boxplots_por_clase.png')


### 2.5 Matriz de correlación

In [ ]:
# Codificación temporal de variables categóricas solo para calcular correlación
df_corr = df.copy()
df_corr['fan_num'] = (df_corr['Fan_Mailing_List'] == 'Yes').astype(int)
df_corr['seat_num'] = (df_corr['Seat_Location'] == 'Lower').astype(int)

# Variables numéricas para la matriz de correlación
cols_num = ['Age', 'Num_Tickets_Purchased', 'Concession_Purchases', 'fan_num', 'seat_num']
corr = df_corr[cols_num].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Matriz de Correlación de Pearson — Predictores', fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/03_heatmap_correlacion.png', bbox_inches='tight')
plt.show()
print('Guardado: 03_heatmap_correlacion.png')


### 2.6 Violin plots y barras de predictores clave


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Violin plot de Num_Tickets_Purchased
sns.violinplot(
    x='Customer_Type', y='Num_Tickets_Purchased', data=df,
    order=orden_clases, palette=paleta, ax=axes[0], inner='box'
)
axes[0].set_title('Num_Tickets_Purchased por tipo de cliente')
axes[0].set_xlabel('Tipo de cliente')
axes[0].set_ylabel('Número de boletas')

# Fan_Mailing_List: ¿los fans planifican más?
fan_counts = df.groupby(['Customer_Type', 'Fan_Mailing_List']).size().reset_index(name='count')
sns.barplot(
    data=fan_counts, x='Customer_Type', y='count',
    hue='Fan_Mailing_List', order=orden_clases,
    palette={'Yes': '#9C27B0', 'No': '#BDBDBD'}, ax=axes[1]
)
axes[1].set_title('Fan Mailing List por tipo de cliente')
axes[1].set_xlabel('Tipo de cliente')
axes[1].set_ylabel('Número de clientes')
axes[1].legend(title='Fan')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/04_violin_dias_fan.png', bbox_inches='tight')
plt.show()
print('Guardado: 04_violin_dias_fan.png')


## 3. Preprocesamiento

Preparamos los datos para PyMC siguiendo el mismo principio del cuaderno original:
- Variables continuas → estandarizar con `StandardScaler`
- Variables binarias → codificar como 0/1
- Variable objetivo → codificar como enteros 0, 1, 2

En esta versión trabajamos con cinco predictores para mantener un modelo más rico que el reducido, pero evitando la variable más tautológica (`Days_Before_Concierto`).


In [ ]:
# ---- CODIFICACIÓN DE VARIABLES CATEGÓRICAS ----

df['fan_int'] = (df['Fan_Mailing_List'] == 'Yes').astype(int)
df['seat_lower'] = (df['Seat_Location'] == 'Lower').astype(int)

# ---- CODIFICACIÓN DE LA VARIABLE OBJETIVO ----
mapa_clases = {'Planner': 0, 'In-Between': 1, 'Last-Minute': 2}
df['y_encoded'] = df['Customer_Type'].map(mapa_clases)

print('Verificación de codificación de la variable objetivo:')
print(df[['Customer_Type', 'y_encoded']].value_counts().sort_index())

# ---- ESTANDARIZACIÓN DE VARIABLES CONTINUAS ----
predictores_continuos = [
    'Age',
    'Num_Tickets_Purchased',
    'Concession_Purchases'
]

scaler = StandardScaler()
X_continuas_z = scaler.fit_transform(df[predictores_continuos])
X_continuas_df = pd.DataFrame(
    X_continuas_z,
    columns=[f'{c}_z' for c in predictores_continuos]
)

predictores_binarios = ['fan_int', 'seat_lower']

X_df = pd.concat(
    [X_continuas_df, df[predictores_binarios].reset_index(drop=True)],
    axis=1
)

X_data = X_df.values.astype('float64')
y_data = df['y_encoded'].values

n_obs       = X_data.shape[0]
n_pred      = X_data.shape[1]
n_clases    = 3

print(f'\nDimensiones del problema:')
print(f'  Observaciones (n):  {n_obs}')
print(f'  Predictores (p):    {n_pred}  → {X_df.columns.tolist()}')
print(f'  Clases (C):         {n_clases} → {list(mapa_clases.keys())}')
print(f'  Parámetros totales: {n_clases} interceptos + {n_pred}×{n_clases} coeficientes = {n_clases + n_pred*n_clases}')

print(f'\nEstadísticas de X estandarizado:')
display(X_df.describe().round(3))


### 3.1 División train/test estratificada

Separamos el 20% de los datos como conjunto de prueba para evaluar el modelo más adelante. `stratify=y_data` garantiza que la proporción de cada clase sea igual en train y test.

In [ ]:
# train_test_split: divide aleatoriamente los datos en dos conjuntos
# test_size=0.2 → 20% para evaluación, 80% para entrenamiento del modelo bayesiano
# stratify=y_data → mantiene la proporción de clases en ambos conjuntos
# random_state=42 → reproducibilidad (siempre la misma división)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data,
    test_size=0.2,
    stratify=y_data,
    random_state=42
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} observaciones')
print(f'Conjunto de prueba:        {X_test.shape[0]} observaciones')
print(f'\nDistribución de clases en train:')
clases_nombre = {v: k for k, v in mapa_clases.items()}
for c in [0, 1, 2]:
    n = (y_train == c).sum()
    print(f'  {clases_nombre[c]:12s}: {n} ({n/len(y_train)*100:.1f}%)')
print(f'\nDistribución de clases en test:')
for c in [0, 1, 2]:
    n = (y_test == c).sum()
    print(f'  {clases_nombre[c]:12s}: {n} ({n/len(y_test)*100:.1f}%)')

## 4. Modelo Bayesiano con PyMC

### Especificación matemática completa (con clase de referencia)

Usamos una parametrización identificable fijando `Planner` como base.

**Priors más informativos:**
$$\alpha_{raw,c} \sim N(0, 1.5^2), \quad c \in \{\text{In-Between},\text{Last-Minute}\}$$
$$\beta_{raw,p,c} \sim N(0, 1.5^2), \quad p=1,\ldots,5$$

Definimos:
$$\alpha = [0,\alpha_{raw,1},\alpha_{raw,2}]$$
$$\beta = [\mathbf{0},\beta_{raw}]$$

donde la primera clase (`Planner`) queda anclada en cero.

Luego:
$$\eta_i = X_i\beta + \alpha, \qquad p_i = \text{softmax}(\eta_i), \qquad Y_i \sim \text{Categorical}(p_i)$$

### ¿Por qué esta parametrización?
Porque elimina la no-identificabilidad aditiva del softmax (desplazamientos comunes de logits), mejora la estabilidad del muestreo y hace que los coeficientes sean interpretables como efectos **relativos a Planner**.


In [ ]:
# Modelo multinomial con clase de referencia (Planner)
with pm.Model() as modelo_multinomial:

    # Solo dos interceptos libres: In-Between y Last-Minute (Planner=0)
    alpha_raw = pm.Normal(
        'alpha_raw',
        mu=0,
        sigma=1.5,
        shape=2
    )

    # Concatenamos la clase base fija en 0
    alpha = pm.Deterministic(
        'alpha',
        pt.concatenate([pt.zeros(1), alpha_raw])
    )

    # Solo dos columnas de coeficientes libres (n_pred x 2)
    beta_raw = pm.Normal(
        'beta_raw',
        mu=0,
        sigma=1.5,
        shape=(n_pred, 2)
    )

    zeros = pt.zeros((n_pred, 1))
    beta = pm.Deterministic(
        'beta',
        pt.concatenate([zeros, beta_raw], axis=1)
    )

    # Logits y softmax
    mu = pm.math.dot(X_train, beta) + alpha
    p = pm.math.softmax(mu, axis=-1)

    # Verosimilitud categórica
    y_obs = pm.Categorical('y_obs', p=p, observed=y_train)

print('Modelo especificado correctamente.')
print(f'Variables libres: {[rv.name for rv in modelo_multinomial.free_RVs]}')
print(f'Variables observadas: {[rv.name for rv in modelo_multinomial.observed_RVs]}')


### 4.1 Muestreo MCMC — NUTS

Usamos el algoritmo **NUTS** (No-U-Turn Sampler), igual que en el cuaderno original.

**Parámetros:**
- `draws=50000`: número de muestras a guardar **por cadena**
- `tune=5000`: pasos de adaptación
- `chains=4`: 4 cadenas independientes
- `target_accept=0.9`: tasa de aceptación objetivo
- `random_seed=42`: reproducibilidad

> ⏱ **Nota:** Si estás explorando rápido en VS Code, puedes bajar temporalmente `draws` y `tune`, pero conviene dejar la configuración larga como referencia final.


In [ ]:
# Muestreo MCMC (NUTS)
with modelo_multinomial:
    trace = pm.sample(
        draws=50000,
        tune=5000,
        chains=4,
        target_accept=0.9,
        random_seed=42,
        return_inferencedata=True,
        progressbar=True
    )

print('\n✓ Muestreo MCMC completado.')


## 5. Diagnósticos MCMC

Antes de interpretar los coeficientes, **debemos verificar que las cadenas convergieron**. Un modelo que no converge da resultados incorrectos aunque no genere errores.

### Criterios de convergencia (del checklist del profesor):
1. **R-hat ≈ 1.0**: las 4 cadenas muestrearon la misma distribución
2. **Trazas estacionarias**: las cadenas mezclan bien sin tendencias
3. **Energy plot**: sin diferencias grandes entre transición y marginal de energía
4. **Sin divergencias**: no hay regiones del espacio posterior inaccesibles

### 5.1 Traceplot

In [ ]:
# Traceplot de los interceptos libres (clases relativas a Planner)
az.plot_trace(
    trace,
    var_names=['alpha_raw'],
    figsize=(12, 5)
)
plt.suptitle('Traceplot — Interceptos libres alpha_raw (vs Planner)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/05_traceplot_alpha.png', bbox_inches='tight')
plt.show()
print('Guardado: 05_traceplot_alpha.png')


In [ ]:
# Traceplot de coeficientes libres beta_raw (n_pred x 2)
az.plot_trace(
    trace,
    var_names=['beta_raw'],
    compact=True,
    figsize=(14, 5)
)
plt.suptitle('Traceplot — Coeficientes libres beta_raw (vs Planner)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/06_traceplot_beta.png', bbox_inches='tight')
plt.show()
print('Guardado: 06_traceplot_beta.png')


### 5.2 R-hat y ESS

In [ ]:
# Diagnóstico de convergencia sobre parámetros libres
resumen = az.summary(trace, var_names=['alpha_raw', 'beta_raw'], round_to=4)
print('Tabla de diagnósticos de convergencia (parámetros libres):')
display(resumen[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk', 'ess_tail']])

rhat_max = resumen['r_hat'].max()
ess_min  = resumen['ess_bulk'].min()

if rhat_max < 1.01:
    print(f'\n✓ Convergencia excelente: R-hat máximo = {rhat_max:.4f} (< 1.01)')
elif rhat_max < 1.05:
    print(f'\n⚠ Convergencia aceptable: R-hat máximo = {rhat_max:.4f} (< 1.05)')
else:
    print(f'\n✗ ADVERTENCIA: R-hat máximo = {rhat_max:.4f} — convergencia insuficiente')

print(f'ESS bulk mínimo: {ess_min:.0f} muestras efectivas')


### 5.3 Energy Plot y divergencias

In [ ]:
# az.plot_energy: compara la distribución de energía de las transiciones (barras)
# con la distribución marginal de energía (línea)
# Si ambas distribuciones son similares → el sampler explora bien el espacio posterior
# Si difieren mucho → hay regiones inaccesibles (patología en el modelo)
az.plot_energy(trace, figsize=(8, 4))
plt.title('Energy Plot — Exploración del Espacio Posterior')
plt.savefig(f'{IMG_DIR}/07_energy_plot.png', bbox_inches='tight')
plt.show()
print('Guardado: 07_energy_plot.png')

# Contar divergencias: ocurren cuando NUTS no puede seguir la curvatura del posterior
# Causas comunes: modelo mal especificado, priors demasiado estrechos, colinealidad extrema
# Muchas divergencias (> 1% del total) indican resultados no confiables
n_div = int(trace.sample_stats.diverging.sum())
total_samples = 50000 * 4  # draws × chains
pct_div = n_div / total_samples * 100

if n_div == 0:
    print(f'\n✓ Sin divergencias (0 de {total_samples:,} muestras)')
elif pct_div < 1:
    print(f'\n⚠ {n_div} divergencias ({pct_div:.2f}% del total) — revisar modelo')
else:
    print(f'\n✗ {n_div} divergencias ({pct_div:.2f}% del total) — modelo problemático')

## 6. Resultados Posteriores

### 6.1 Distribuciones posteriores de los interceptos

Los interceptos `alpha[c]` representan el log-odds de pertenecer a la clase $c$ cuando **todos los predictores son cero** (es decir, cuando el cliente es "promedio" en todas las variables estandarizadas y las binarias son 0).

In [ ]:
# Posterior de interceptos libres
az.plot_posterior(
    trace,
    var_names=['alpha_raw'],
    hdi_prob=0.95,
    ref_val=0,
    figsize=(12, 4),
    textsize=10
)

for ax, label in zip(plt.gcf().axes, ['alpha_raw[0] — In-Between vs Planner', 'alpha_raw[1] — Last-Minute vs Planner']):
    ax.set_title(label)

plt.suptitle('Distribuciones Posteriores — Interceptos libres (HDI 95%)', y=1.05, fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/08_posterior_interceptos.png', bbox_inches='tight')
plt.show()
print('Guardado: 08_posterior_interceptos.png')


### 6.2 Forest Plot de coeficientes por clase

El forest plot muestra simultáneamente los intervalos HDI de todos los coeficientes. Es la visualización clave para comparar la magnitud e incertidumbre de cada predictor.

**Interpretación:** Si el HDI de `beta[j, c]` **no cruza el cero**, hay evidencia sólida de que el predictor $j$ discrimina a los clientes hacia/contra la clase $c$.

In [ ]:
# Forest de coeficientes libres (comparaciones relativas a Planner)
az.plot_forest(
    trace,
    var_names=['beta_raw'],
    combined=True,
    hdi_prob=0.95,
    r_hat=True,
    figsize=(10, 10)
)
plt.axvline(0, color='red', linestyle='--', linewidth=1.2, label='Efecto nulo (β=0)')
plt.title('Forest Plot — beta_raw (HDI 95%)\nClase 0=In-Between vs Planner, Clase 1=Last-Minute vs Planner', fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/09_forest_plot_beta.png', bbox_inches='tight')
plt.show()
print('Guardado: 09_forest_plot_beta.png')


In [ ]:
# Forest de interceptos libres
az.plot_forest(
    trace,
    var_names=['alpha_raw'],
    combined=True,
    hdi_prob=0.95,
    r_hat=True,
    figsize=(8, 3)
)
plt.axvline(0, color='red', linestyle='--', linewidth=1.2)
plt.title('Forest Plot — alpha_raw (HDI 95%)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/10_forest_plot_alpha.png', bbox_inches='tight')
plt.show()
print('Guardado: 10_forest_plot_alpha.png')


### 6.3 Tabla resumen de coeficientes con nombres de predictores

In [ ]:
# Construimos una tabla legible con los nombres de los predictores
# para facilitar la interpretación del forest plot

nombres_pred = X_df.columns.tolist()
nombres_clases = list(mapa_clases.keys())

beta_posterior = trace.posterior['beta']
beta_mean = beta_posterior.mean(dim=['chain', 'draw']).values

tabla_coef = pd.DataFrame(
    beta_mean,
    index=nombres_pred,
    columns=[f'β — {c}' for c in nombres_clases]
)

print('Media posterior de coeficientes beta por clase:')
print('(valores positivos = mayor probabilidad de esa clase, negativos = menor probabilidad)')
display(tabla_coef.round(3))

efecto_abs = tabla_coef.abs().mean(axis=1)
pred_mayor = efecto_abs.idxmax()
print(f'\nPredictor con mayor efecto medio: {pred_mayor} (efecto abs. promedio = {efecto_abs[pred_mayor]:.3f})')


## 7. Posterior Predictive Check (PPC)

El PPC verifica si el modelo puede regenerar datos similares a los observados. Siguiendo el estilo del profesor (como en `RegresionLineal.ipynb` con `pm.sample_posterior_predictive`), generamos predicciones sobre el conjunto de prueba.

In [ ]:
# PPC en test usando la misma parametrización con referencia
with pm.Model() as modelo_pred:
    alpha_raw_pred = pm.Normal('alpha_raw', mu=0, sigma=1.5, shape=2)
    alpha_pred = pm.Deterministic('alpha', pt.concatenate([pt.zeros(1), alpha_raw_pred]))

    beta_raw_pred = pm.Normal('beta_raw', mu=0, sigma=1.5, shape=(n_pred, 2))
    zeros_pred = pt.zeros((n_pred, 1))
    beta_pred = pm.Deterministic('beta', pt.concatenate([zeros_pred, beta_raw_pred], axis=1))

    mu_pred = pm.math.dot(X_test, beta_pred) + alpha_pred
    p_pred = pm.math.softmax(mu_pred, axis=-1)
    y_obs_pred = pm.Categorical('y_obs', p=p_pred, observed=y_test)

    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

print('✓ Posterior Predictive Check completado.')


### 7.1 Probabilidades predichas por clase

In [ ]:
# Las predicciones del PPC son muestras de la distribución predictiva posterior
# ppc.posterior_predictive['y_obs'] tiene forma (chains, draws, n_test)
# Cada valor es una clase predicha (0, 1 o 2) muestreada del posterior

# Para obtener probabilidades por clase, calculamos la frecuencia con que
# el modelo predice cada clase para cada observación del test set

# y_pred_samples tiene forma (chains*draws, n_test)
y_pred_samples = ppc.posterior_predictive['y_obs'].values.reshape(-1, len(y_test))

# Calcular probabilidad de cada clase para cada observación
# prob[i, c] = fracción de muestras donde el modelo predijo clase c para la observación i
prob_pred = np.stack([
    (y_pred_samples == c).mean(axis=0)  # promedio sobre todas las muestras MCMC
    for c in range(n_clases)
], axis=1)  # resultado: (n_test, 3)

# La clase predicha es la que tiene mayor probabilidad posterior
y_pred_clase = prob_pred.argmax(axis=1)  # argmax sobre las 3 clases

# DataFrame con probabilidades predichas
df_pred = pd.DataFrame(
    prob_pred,
    columns=[f'P({c})' for c in nombres_clases]
)
df_pred['Clase_real']    = [nombres_clases[c] for c in y_test]
df_pred['Clase_pred']    = [nombres_clases[c] for c in y_pred_clase]
df_pred['Correcta']      = df_pred['Clase_real'] == df_pred['Clase_pred']

print('Muestra de probabilidades predichas (primeras 10 observaciones del test):')
display(df_pred.head(10).round(3))
print(f'\nAccuracy: {df_pred["Correcta"].mean():.4f} ({df_pred["Correcta"].mean()*100:.1f}%)')

### 7.2 Matriz de confusión

La matriz de confusión muestra cuántas predicciones fueron correctas (diagonal) y cuáles se confundieron entre clases (fuera de la diagonal). Es la misma métrica que el profesor usa en `Modelos_Multivariados_2.ipynb` para el clasificador spam/no-spam.

In [ ]:
# confusion_matrix: matriz n_clases × n_clases
# conf[i, j] = número de observaciones reales de clase i predichas como clase j
# La diagonal principal (conf[i, i]) = predicciones correctas
conf = confusion_matrix(y_test, y_pred_clase)

# Visualizar con heatmap anotado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de conteos absolutos
sns.heatmap(
    conf,
    annot=True,           # mostrar números en cada celda
    fmt='d',              # formato entero
    cmap='Blues',         # escala de azules: más oscuro = más observaciones
    xticklabels=nombres_clases,
    yticklabels=nombres_clases,
    ax=axes[0]
)
axes[0].set_title('Matriz de Confusión (conteos)')
axes[0].set_xlabel('Clase predicha')
axes[0].set_ylabel('Clase real')

# Heatmap de proporciones (normalizado por fila = recall por clase)
conf_norm = conf.astype(float) / conf.sum(axis=1, keepdims=True)  # normalizar por fila
sns.heatmap(
    conf_norm,
    annot=True,
    fmt='.2%',            # formato porcentaje
    cmap='Blues',
    xticklabels=nombres_clases,
    yticklabels=nombres_clases,
    ax=axes[1]
)
axes[1].set_title('Matriz de Confusión (% por fila = Recall)')
axes[1].set_xlabel('Clase predicha')
axes[1].set_ylabel('Clase real')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/11_matriz_confusion.png', bbox_inches='tight')
plt.show()
print('Guardado: 11_matriz_confusion.png')

# Reporte de clasificación completo (igual que el profe en spam)
# Incluye: precision, recall, F1-score y soporte por clase
print('\nReporte de Clasificación:')
print(classification_report(y_test, y_pred_clase, target_names=nombres_clases))

### 7.3 Visualización de probabilidades predichas por clase real

In [ ]:
# Scatter de probabilidades predichas: cada punto es un cliente del test set
# Eje X: probabilidad predicha de ser Planner
# Eje Y: probabilidad predicha de ser Last-Minute
# Color: clase real del cliente
# Si el modelo discrimina bien, las clases deberían separarse en el espacio de probabilidades

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_clase = {'Planner': '#2196F3', 'In-Between': '#FF9800', 'Last-Minute': '#4CAF50'}

# Scatter: P(Planner) vs P(Last-Minute)
for clase in nombres_clases:
    mask = df_pred['Clase_real'] == clase
    axes[0].scatter(
        df_pred.loc[mask, 'P(Planner)'],
        df_pred.loc[mask, 'P(Last-Minute)'],
        alpha=0.5, label=clase,
        color=colores_clase[clase],
        edgecolors='white', linewidths=0.3, s=40
    )
axes[0].set_xlabel('P(Planner)')
axes[0].set_ylabel('P(Last-Minute)')
axes[0].set_title('Probabilidades predichas: Planner vs Last-Minute')
axes[0].legend()

# Stacked bar: probabilidad promedio predicha por clase real
prob_por_clase_real = df_pred.groupby('Clase_real')[['P(Planner)', 'P(In-Between)', 'P(Last-Minute)']].mean()
prob_por_clase_real.plot(
    kind='bar', stacked=True,
    color=['#2196F3', '#FF9800', '#4CAF50'],
    ax=axes[1], edgecolor='white'
)
axes[1].set_title('Probabilidad media predicha por clase real')
axes[1].set_xlabel('Clase real')
axes[1].set_ylabel('Probabilidad promedio')
axes[1].legend(title='Clase predicha')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/12_probabilidades_predichas.png', bbox_inches='tight')
plt.show()
print('Guardado: 12_probabilidades_predichas.png')

## 8. Perfiles de Compradores

Respondemos las preguntas orientadoras del taller basándonos en los coeficientes posteriores del **modelo con 5 variables**.


In [ ]:
# Heatmap de coeficientes medios: muestra el efecto de cada predictor en cada clase
# Colores cálidos (rojo) = coeficiente positivo → mayor probabilidad de esa clase
# Colores fríos (azul) = coeficiente negativo → menor probabilidad de esa clase

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    tabla_coef.T,           # transpuesta: clases en filas, predictores en columnas
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',          # divergente: rojo=positivo, azul=negativo
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Media Posterior de Coeficientes β por Clase y Predictor', fontsize=12)
ax.set_xlabel('Predictor')
ax.set_ylabel('Clase')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/13_heatmap_coeficientes.png', bbox_inches='tight')
plt.show()
print('Guardado: 13_heatmap_coeficientes.png')

In [ ]:
# ---- RESUMEN INTERPRETATIVO AUTOMÁTICO ----

def get_beta_stats(pred_idx, clase_idx):
    """Retorna (media, hdi_low, hdi_high) de beta[pred, clase]."""
    vals = trace.posterior['beta'].values[:, :, pred_idx, clase_idx].flatten()
    mean_val = float(vals.mean())
    hdi = az.hdi(vals, hdi_prob=0.95)
    return mean_val, float(hdi[0]), float(hdi[1])

def cruza_cero(low, high):
    return low < 0 < high

idx_age = nombres_pred.index('Age_z')
idx_tickets = nombres_pred.index('Num_Tickets_Purchased_z')
idx_conc = nombres_pred.index('Concession_Purchases_z')
idx_fan = nombres_pred.index('fan_int')
idx_seat = nombres_pred.index('seat_lower')

print('=' * 72)
print('PERFILES DE COMPRADORES — REGRESIÓN MULTINOMIAL BAYESIANA')
print('Movistar Arena — Modelo 5 variables con clase de referencia')
print('=' * 72)
print('Referencia fija: Planner (todos sus coeficientes = 0 por construcción).')

for c_idx, c_nombre in enumerate(nombres_clases):
    print(f'\n{"─"*70}')
    print(f'CLASE {c_idx}: {c_nombre.upper()}')
    print(f'{"─"*70}')
    if c_idx == 0:
        print('  Clase de referencia: coeficientes fijados en 0 (sin incertidumbre).')
        continue
    for p_idx, p_nombre in enumerate(nombres_pred):
        m, lo, hi = get_beta_stats(p_idx, c_idx)
        conclusion = '✓ Efecto robusto' if not cruza_cero(lo, hi) else '⚠ Efecto incierto'
        print(f'  {p_nombre:<30s}: β={m:+.3f}  [HDI95%: {lo:.3f}, {hi:.3f}]  {conclusion}')

print(f'\n{"═"*72}')
print('RESPUESTAS A LAS PREGUNTAS ORIENTADORAS DEL TALLER')
print(f'{"═"*72}')

m_tick_ib, lo_tick_ib, hi_tick_ib = get_beta_stats(idx_tickets, 1)
m_tick_lm, lo_tick_lm, hi_tick_lm = get_beta_stats(idx_tickets, 2)
print(f"""
1. ¿Qué caracteriza Planner frente a las demás clases?
   Planner es la referencia; los coeficientes estimados indican cuánto se alejan
   In-Between y Last-Minute respecto a Planner para cada predictor.
   - In-Between vs Planner (Num_Tickets): β={m_tick_ib:+.3f}
   - Last-Minute vs Planner (Num_Tickets): β={m_tick_lm:+.3f}
""")

m_fan_ib, lo_fan_ib, hi_fan_ib = get_beta_stats(idx_fan, 1)
m_fan_lm, lo_fan_lm, hi_fan_lm = get_beta_stats(idx_fan, 2)
print(f"""2. Señales de compra tardía y afinidad con la marca:
   - In-Between vs Planner (fan_int):   β={m_fan_ib:+.3f}
   - Last-Minute vs Planner (fan_int):  β={m_fan_lm:+.3f}
""")

print(f"""3. Predictor más discriminante (magnitud media):
   → {pred_mayor}: efecto absoluto promedio = {efecto_abs[pred_mayor]:.3f}
""")

accuracy = df_pred['Correcta'].mean()
print(f"""4. Desempeño predictivo:
   Accuracy en test: {accuracy*100:.1f}%
   Esta parametrización evita no-identificabilidad del softmax y hace que los HDI
   se interpreten de forma más estable en términos relativos a Planner.
""")

print('=' * 72)


## Resumen de figuras generadas

Todas las figuras se guardaron en `img/resultados_movistar_5vars_ref/`:


In [ ]:
import glob
figuras = sorted(glob.glob(f'{IMG_DIR}/*.png'))
print(f'Figuras generadas en {IMG_DIR}/ ({len(figuras)} archivos):')
for f in figuras:
    print(f'  {os.path.basename(f)}')